
# Round7 — Real-World Dataset Builder

이번 단계의 목적은 **실제 서로 다른 대피안내도 원본 수를 늘리는 것**입니다.

- 공식 대학/공공기관/의료기관의 공개 evacuation-map PDF를 다운로드
- 실제 evacuation map이 있는 페이지만 자동 추출
- PNG 원본으로 변환
- 기존 `photos_all` 계열이 있으면 변형본 전체가 아니라 **원본 계열 대표 이미지**만 추가
- 중복 제거
- 현재 Round6 모델이 있으면 실사진에 추론해 일반화 상태를 먼저 확인

> 외부 공개 자료의 재사용/상업적 이용 권리는 출처별로 다릅니다. 이 노트북은 원본을 패키지에 재배포하지 않고 공식 URL에서 직접 내려받습니다.


In [ ]:

# CELL 1 — 설치 + Drive + 경로

!pip -q install pymupdf pillow imagehash requests ultralytics

from google.colab import drive, files
from pathlib import Path
import csv, json, shutil, re, os, hashlib

drive.mount("/content/drive")

OUT = Path("/content/drive/MyDrive/evacuation_yolo/round7_realworld")
OUT.mkdir(parents=True, exist_ok=True)

PDF_DIR = OUT/"source_pdfs"
EXT_DIR = OUT/"external_originals"
EXISTING_DIR = OUT/"existing_real_originals"
FINAL_DIR = OUT/"real_originals"
PRED_DIR = OUT/"predictions"

for p in [PDF_DIR, EXT_DIR, EXISTING_DIR, FINAL_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Output:", OUT)


In [ ]:

# CELL 2 — 공식 공개 evacuation-map PDF 다운로드

import requests, csv
from pathlib import Path

SOURCES = [
    ("scu_bannan","Santa Clara University",
     "https://university-operations.scu.edu/media/offices/university-operations/building-images/building-eaps/Bannan-Eng-Labs-Evac-Maps.pdf"),
    ("ucalgary_aurora","University of Calgary",
     "https://www.ucalgary.ca/risk/sites/default/files/teams/22/aurora-hall-evac-maps.pdf"),
    ("nkust","NKUST",
     "https://shecenter.nkust.edu.tw/var/file/23/1023/img/1267/579998647.pdf"),
    ("utm_m50","Universiti Teknologi Malaysia",
     "https://civil.utm.my/osh/wp-content/uploads/sites/2514/2026/05/MAKMAL-STRUKTUR-BLOK-M50.pdf"),
    ("kfupm_pegtc","KFUPM",
     "https://cpg.kfupm.edu.sa/wp-content/uploads/2024/12/Evacualtion-Plan-All-Floors-PEGTC.pdf"),
    ("imperial_county","Imperial County BHS",
     "https://bhs-intranet.imperialcounty.org/wp-content/uploads/2023/04/Emergency-Evacuation-Plan-2022_23_-2_27_2023.pdf"),
    ("stanford_gse","Stanford GSE",
     "https://ed.stanford.edu/sites/default/files/facility/graduate_school_of_education_emergency_plan.pdf"),
    ("beebe_rollins","Beebe Healthcare",
     "https://www.beebehealthcare.org/sites/default/files/inline-images/n9LJhAnv3a9imUkHrJxjbJ4zvhwjBoijKtEFb55DOv0Mch7z1e.pdf"),
]

session=requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0 Round7Research/1.0"})

download_report=[]

for sid, institution, url in SOURCES:
    dst=PDF_DIR/f"{sid}.pdf"
    try:
        r=session.get(url,timeout=60)
        r.raise_for_status()
        if not r.content.startswith(b"%PDF"):
            raise RuntimeError("response is not PDF")
        dst.write_bytes(r.content)
        status="ok"
        size=len(r.content)
    except Exception as e:
        status=f"error: {e}"
        size=0
    download_report.append({
        "id":sid,"institution":institution,"url":url,
        "status":status,"bytes":size
    })
    print("✅" if status=="ok" else "❌", sid, status)

with open(OUT/"source_download_report.csv","w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=download_report[0].keys())
    w.writeheader(); w.writerows(download_report)

print("\nDownloaded:",sum(x["status"]=="ok" for x in download_report),"/",len(download_report))


In [ ]:

# CELL 3 — evacuation map 페이지 자동 탐색 + PNG 추출

import fitz
from PIL import Image
from pathlib import Path
import csv

KEYWORDS = [
    "evacuation", "you are here", "fire extinguisher", "exit",
    "stair", "elevator", "hydrant", "escape route", "emergency"
]

extract_rows=[]

for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    sid=pdf_path.stem
    doc=fitz.open(pdf_path)
    selected=[]

    for i,page in enumerate(doc):
        text=(page.get_text("text") or "").lower()
        hits=sum(1 for k in KEYWORDS if k in text)

        # map page는 보통 여러 관련 키워드를 함께 포함.
        if hits >= 2:
            selected.append((i,hits))

    # 텍스트 추출이 약한 스캔 PDF는 첫 몇 페이지라도 후보로 남김
    if not selected and len(doc) <= 10:
        selected=[(i,0) for i in range(len(doc))]

    # 지나치게 큰 문서는 최대 12페이지까지만
    selected=selected[:12]

    for page_idx,hits in selected:
        page=doc[page_idx]
        pix=page.get_pixmap(matrix=fitz.Matrix(2.0,2.0), alpha=False)
        out=EXT_DIR/f"{sid}_p{page_idx+1:03d}.png"
        pix.save(str(out))
        extract_rows.append({
            "source_id":sid,
            "page":page_idx+1,
            "keyword_hits":hits,
            "image":str(out),
            "source_pdf":str(pdf_path)
        })

    print(f"{sid}: {len(selected)} page(s) extracted / {len(doc)} total")

with open(OUT/"external_extract_manifest.csv","w",newline="",encoding="utf-8") as f:
    fields=["source_id","page","keyword_hits","image","source_pdf"]
    w=csv.DictWriter(f,fieldnames=fields)
    w.writeheader(); w.writerows(extract_rows)

print("\n✅ external originals:",len(extract_rows))


In [ ]:

# CELL 4 — 기존 실제 원본 계열 대표 이미지 수집
# photos_all이 있으면 v00/original 우선, 없으면 source-family 당 1장만 선택

from pathlib import Path
import shutil, re

known_candidates = [
    Path("/content/drive/MyDrive/02/ml/real_data/photos_all"),
    Path("/content/drive/MyDrive/02/ML/real_data/photos_all"),
]

photos_all = next((p for p in known_candidates if p.exists()), None)

if photos_all is None:
    print("ℹ️ 기존 photos_all 폴더를 찾지 못했습니다. 외부 실제 원본만 사용합니다.")
else:
    imgs=[p for p in photos_all.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}]

    families={}
    for p in imgs:
        m=re.search(r"(evac_src\d+)",p.stem,re.I)
        fam=m.group(1).lower() if m else re.sub(r"_v\d+.*$","",p.stem.lower())
        score=0
        s=p.stem.lower()
        if "_v00" in s: score+=10
        if "original" in s: score+=5
        if fam not in families or score>families[fam][0]:
            families[fam]=(score,p)

    for fam,(_,src) in families.items():
        dst=EXISTING_DIR/f"{fam}_{src.name}"
        shutil.copy2(src,dst)

    print("✅ 기존 실제 source-family 대표:",len(families))


In [ ]:

# CELL 5 — perceptual hash로 중복 제거 + 통합 real_originals 생성

from PIL import Image
import imagehash, shutil, csv
from pathlib import Path

if FINAL_DIR.exists():
    shutil.rmtree(FINAL_DIR)
FINAL_DIR.mkdir(parents=True)

candidates = []
for origin,folder in [("external",EXT_DIR),("existing",EXISTING_DIR)]:
    for p in folder.glob("*"):
        if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}:
            candidates.append((origin,p))

kept=[]
manifest=[]
hashes=[]

for origin,p in sorted(candidates,key=lambda x:str(x[1])):
    try:
        with Image.open(p) as im:
            h=imagehash.phash(im.convert("RGB").resize((1024,1024)))
    except Exception:
        continue

    duplicate=False
    for old_h,_ in hashes:
        if h-old_h <= 3:
            duplicate=True
            break
    if duplicate:
        continue

    hashes.append((h,p))
    dst=FINAL_DIR/f"{origin}_{len(kept):04d}_{p.name}"
    shutil.copy2(p,dst)
    kept.append(dst)
    manifest.append({
        "index":len(kept)-1,
        "origin":origin,
        "filename":dst.name,
        "path":str(dst),
        "phash":str(h),
    })

with open(OUT/"real_originals_manifest.csv","w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=manifest[0].keys() if manifest else ["index","origin","filename","path","phash"])
    w.writeheader()
    if manifest: w.writerows(manifest)

print("======================================")
print("✅ REAL ORIGINAL POOL READY")
print("======================================")
print("External extracted:",len(list(EXT_DIR.glob("*"))))
print("Existing originals :",len(list(EXISTING_DIR.glob("*"))))
print("After dedupe       :",len(kept))
print("Folder             :",FINAL_DIR)


In [ ]:

# CELL 6 — 현재 Round6 모델로 실제 원본 inference (학습 전 generalization 점검)
# Round6 class_router.json + 모델이 있으면 자동 실행.
# 없으면 이 셀은 건너뛰어도 됩니다.

from ultralytics import YOLO
from pathlib import Path
import json, cv2
from collections import defaultdict

ROUND6 = Path("/content/drive/MyDrive/evacuation_yolo/round6_allclass_recall")
router_path = ROUND6/"class_router.json"

if not router_path.exists():
    print("ℹ️ Round6 class_router.json이 없어 inference를 건너뜁니다.")
else:
    router=json.loads(router_path.read_text(encoding="utf-8"))

    # router의 Colab 경로가 현재 런타임과 다를 수 있으므로 모델 이름으로 재해석
    model_paths = {
        "round2": Path("/content/round6_allclass_recall/models/round2_best.pt"),
        "round4": Path("/content/round6_allclass_recall/models/round4_guarded_best.pt"),
        "stage_a": ROUND6/"runs/stage_a/weights/best.pt",
        "stage_b": ROUND6/"runs/stage_b/weights/best.pt",
    }

    needed=sorted(set(r["model_name"] for r in router.values() if r["model_name"]!="UNAVAILABLE"))
    missing=[m for m in needed if not model_paths.get(m,Path("/none")).exists()]

    if missing:
        print("⚠️ 필요한 모델 파일이 없습니다:",missing)
        print("Round6 패키지/가중치를 준비한 뒤 다시 실행하세요.")
    else:
        models={name:YOLO(str(model_paths[name])) for name in needed}
        grouped=defaultdict(list)
        for cname,r in router.items():
            if r["model_name"]!="UNAVAILABLE":
                grouped[r["model_name"]].append((r["class_id"],float(r["threshold"]),cname))

        all_results=[]
        for img_path in sorted(FINAL_DIR.glob("*")):
            im=cv2.imread(str(img_path))
            if im is None: continue
            drawn=im.copy()
            per_image=[]

            for model_name,specs in grouped.items():
                min_conf=min(t for _,t,_ in specs)
                class_ids=[cid for cid,_,_ in specs]
                thresholds={cid:t for cid,t,_ in specs}
                res=models[model_name].predict(
                    str(img_path),imgsz=896,conf=min_conf,iou=.5,
                    classes=class_ids,verbose=False
                )[0]

                if res.boxes is None: continue
                for xyxy,conf,cid in zip(
                    res.boxes.xyxy.cpu().numpy(),
                    res.boxes.conf.cpu().numpy(),
                    res.boxes.cls.cpu().numpy().astype(int)
                ):
                    if float(conf) < thresholds[int(cid)]:
                        continue
                    cname=["exit","stair","elevator","extinguisher","hydrant","you_are_here","door","room"][int(cid)]
                    x1,y1,x2,y2=map(int,xyxy)
                    cv2.rectangle(drawn,(x1,y1),(x2,y2),(0,255,0),2)
                    cv2.putText(drawn,f"{cname} {conf:.2f}",(x1,max(20,y1-4)),
                                cv2.FONT_HERSHEY_SIMPLEX,.5,(0,255,0),1,cv2.LINE_AA)
                    per_image.append({
                        "class_id":int(cid),"class_name":cname,
                        "confidence":float(conf),"model":model_name,
                        "bbox":[x1,y1,x2,y2]
                    })

            cv2.imwrite(str(PRED_DIR/img_path.name),drawn)
            all_results.append({"image":img_path.name,"detections":per_image})

        (OUT/"round6_realworld_predictions.json").write_text(
            json.dumps(all_results,indent=2,ensure_ascii=False),encoding="utf-8"
        )
        print("✅ inference images:",len(all_results))
        print("Results:",PRED_DIR)



## 다음 단계

이 노트북이 끝나면 `real_originals/`가 **실제 서로 다른 대피안내도 원본 풀**입니다.

외부 원본은 아직 human-GT가 아니므로 바로 fine-tuning에 넣기보다는:
1. Round6 예측 결과 확인
2. 잘못된 박스만 수정하여 GT 확정
3. source 단위로 Train / Val / Test 분리
4. 실제 원본 중심 fine-tuning

순서가 안전합니다.

특히 `exit`, `stair`, `you_are_here`는 안전/경로탐색 핵심 클래스이므로 외부 원본에서 별도 검수하는 것을 권장합니다.
